# Study 02: SQL 평가 - Execution Accuracy 이해하기

**목표**: SQL Agent 평가 방법인 Execution Accuracy를 이해하고, 왜 RAG와 다른 접근이 필요한지 배웁니다.

**소요 시간**: 약 20분

---

## 1. 왜 SQL 평가는 RAG와 다른가?

### RAG vs SQL 평가의 근본적 차이

| 유형 | 평가 대상 | 표준 도구 | 평가 방식 |
|------|----------|----------|----------|
| **RAG** | 텍스트 답변 | RAGAS 라이브러리 | LLM-as-Judge |
| **SQL** | 쿼리 결과 | 직접 구현 | DB 실행 결과 비교 |

### RAG (텍스트 기반) - RAGAS 사용

```
질문: "연차 정책이 뭐야?"
정답: "입사 1년차는 11일, 2년차는 15일의 연차를 받습니다."
생성: "1년차 직원은 11일의 연차가 부여됩니다."

→ 같은 의미, 다른 표현 → LLM이 의미적 유사도를 판단해야 함 (RAGAS)
```

### SQL (실행 기반) - Execution Accuracy

```
질문: "직원 수는?"
정답 SQL: SELECT COUNT(*) FROM employees
생성 SQL: SELECT COUNT(emp_id) FROM employees

→ 다른 SQL 문법, 같은 실행 결과 → DB가 직접 판단 가능!
```

## 2. Execution Accuracy란?

### 핵심 개념

**Execution Accuracy** = "SQL 문법이 달라도, 실행 결과가 같으면 정답"

```python
# 정답 SQL
gold_sql = "SELECT COUNT(*) FROM employees"

# LLM이 생성한 SQL (문법은 다름)
gen_sql = "SELECT COUNT(emp_id) FROM employees"

# 두 쿼리 실행
gold_result = db.execute(gold_sql)  # [(15,)]
gen_result = db.execute(gen_sql)    # [(15,)]

# 결과 비교
is_correct = (gold_result == gen_result)  # True!
```

### 왜 텍스트 비교가 아닌가?

```python
# 텍스트 비교 (Exact Match) - 너무 엄격함!
"SELECT COUNT(*) FROM employees" == "SELECT COUNT(emp_id) FROM employees"  # False

# 하지만 두 쿼리는 동일한 결과를 반환함
# 따라서 Execution Accuracy가 더 적절한 평가 방식
```

## 3. 간단한 예제로 이해하기

환경을 설정하고 Execution Accuracy 개념을 직접 확인해봅시다.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트 설정
PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# .env 파일 로드
load_dotenv(PROJECT_ROOT / ".env")

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"DATABASE_URL 설정: {'OK' if os.environ.get('DATABASE_URL') else 'NOT SET'}")

In [ ]:
from core.database.connection import DatabaseConnection

# 데이터베이스 연결
db = DatabaseConnection(os.environ["DATABASE_URL"])

print("데이터베이스 연결 성공!")

### 예제 1: 같은 결과, 다른 SQL

두 개의 다른 SQL이 같은 결과를 반환하는 경우

In [ ]:
# 방법 1: COUNT(*)
sql_1 = "SELECT COUNT(*) as total FROM employees"
result_1, error_1 = db.execute_query(sql_1)

# 방법 2: COUNT(emp_id)
sql_2 = "SELECT COUNT(emp_id) as total FROM employees"
result_2, error_2 = db.execute_query(sql_2)

print("=== SQL 비교 ===")
print(f"SQL 1: {sql_1}")
print(f"SQL 2: {sql_2}")
print(f"\n텍스트가 같은가? {sql_1 == sql_2}")

print("\n=== 실행 결과 비교 ===")
print(f"결과 1: {result_1}")
print(f"결과 2: {result_2}")
print(f"\n결과가 같은가? {result_1 == result_2}")
print("\n→ SQL 문법은 다르지만, 실행 결과가 같으므로 Execution Accuracy = 정답!")

### 예제 2: 복잡한 JOIN 쿼리

조인 순서나 별칭이 달라도 결과가 같으면 정답

In [ ]:
# 방법 1: 정석적인 JOIN
sql_1 = """
SELECT e.name, d.name as dept_name
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
WHERE d.name = '개발'
ORDER BY e.name
""".strip()

# 방법 2: 다른 스타일의 JOIN
sql_2 = """
SELECT employees.name, departments.name as dept_name
FROM departments
INNER JOIN employees ON departments.dept_id = employees.dept_id
WHERE departments.name = '개발'
ORDER BY employees.name ASC
""".strip()

result_1, _ = db.execute_query(sql_1)
result_2, _ = db.execute_query(sql_2)

print("=== 복잡한 JOIN 비교 ===")
print(f"SQL 1 (e, d 별칭 사용):\n{sql_1}\n")
print(f"SQL 2 (테이블명 전체 사용):\n{sql_2}\n")

print("=== 실행 결과 ===")
print(f"결과 1: {result_1}")
print(f"결과 2: {result_2}")
print(f"\n결과가 같은가? {result_1 == result_2}")

## 4. 현업 표준: defog-ai/sql-eval

### 업계에서 사용하는 SQL 평가 라이브러리

[defog-ai/sql-eval](https://github.com/defog-ai/sql-eval)은 SQL 평가의 사실상 표준입니다.

```python
# defog-ai/sql-eval 사용 예시
from eval.eval import compare_query_results

exact_match, is_correct = compare_query_results(
    query_gold="SELECT author.name, COUNT(*) FROM author...",
    query_gen="SELECT a.name, COUNT(w.pid) FROM author a...",
    db_name="academic"
)

# exact_match = False (SQL 문법이 다름)
# is_correct = True (실행 결과가 같음) ← 이것이 Execution Accuracy!
```

### 핵심 로직 (우리가 직접 구현할 부분)

```python
def compare_query_results(gold_sql, gen_sql, db):
    # 1. 두 쿼리 모두 실행
    gold_result = db.execute(gold_sql)
    gen_result = db.execute(gen_sql)
    
    # 2. 결과 정규화 (순서 무시, 타입 통일)
    gold_normalized = normalize(gold_result)
    gen_normalized = normalize(gen_result)
    
    # 3. 비교
    exact_match = (gold_sql == gen_sql)
    is_correct = (gold_normalized == gen_normalized)  # Execution Accuracy
    
    return exact_match, is_correct
```

## 5. 우리 프로젝트에서 직접 구현하는 이유

### defog-ai/sql-eval의 한계

- PostgreSQL/MySQL 전용 복잡한 설정 필요
- Docker 기반 데이터베이스 설정 필수
- 프로젝트 규모에 비해 과도한 의존성

### 직접 구현의 장점

1. **학습 효과**: Execution Accuracy의 원리를 직접 이해
2. **간단함**: 우리 프로젝트는 단일 MySQL DB만 사용
3. **유연성**: 프로젝트 요구사항에 맞게 커스터마이징 가능

### 핵심 구현 코드 미리보기

```python
def compare_results(result1: List[Dict], result2: List[Dict]) -> bool:
    """두 SQL 실행 결과 비교 (순서 무시)"""
    if result1 is None or result2 is None:
        return False
    
    def normalize(results):
        # Dict를 정렬 가능한 튜플로 변환
        normalized = []
        for row in results:
            sorted_items = tuple(sorted(row.items()))
            normalized.append(sorted_items)
        return sorted(normalized)
    
    return normalize(result1) == normalize(result2)
```

## 6. 결과 비교 함수 직접 테스트

In [ ]:
from typing import List, Dict

def compare_results(result1: List[Dict], result2: List[Dict]) -> bool:
    """
    두 SQL 실행 결과 비교 (Execution Accuracy 핵심 로직)
    
    - 순서 무시 (ORDER BY 없는 경우)
    - 타입 정규화 (int/float 등)
    """
    if result1 is None or result2 is None:
        return False
    
    def normalize(results):
        """결과를 정렬 가능한 형태로 정규화"""
        normalized = []
        for row in results:
            # Dict를 정렬된 튜플로 변환
            sorted_items = tuple(sorted(row.items()))
            normalized.append(sorted_items)
        return sorted(normalized)
    
    return normalize(result1) == normalize(result2)

print("compare_results 함수 정의 완료!")

In [ ]:
# 테스트 케이스 1: 같은 결과
result_a = [{"name": "김철수", "dept": "개발"}, {"name": "이영희", "dept": "인사"}]
result_b = [{"name": "이영희", "dept": "인사"}, {"name": "김철수", "dept": "개발"}]  # 순서만 다름

print("=== 테스트 1: 순서만 다른 결과 ===")
print(f"결과 A: {result_a}")
print(f"결과 B: {result_b}")
print(f"compare_results: {compare_results(result_a, result_b)}")
print("→ 순서가 달라도 내용이 같으면 True\n")

# 테스트 케이스 2: 다른 결과
result_c = [{"name": "김철수", "dept": "개발"}]
result_d = [{"name": "박민수", "dept": "영업"}]

print("=== 테스트 2: 내용이 다른 결과 ===")
print(f"결과 C: {result_c}")
print(f"결과 D: {result_d}")
print(f"compare_results: {compare_results(result_c, result_d)}")
print("→ 내용이 다르면 False")

## 7. SQLAgent와 연동 테스트

실제 SQLAgent가 생성한 SQL과 정답 SQL의 결과를 비교해봅시다.

In [ ]:
from core.agents.sql_agent import SQLAgent

# SQLAgent 생성
agent = SQLAgent(
    db=db,
    model=os.environ.get("OLLAMA_MODEL", "qwen3:8b"),
    provider="ollama",
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
)

print("SQLAgent 생성 완료!")

In [ ]:
# 테스트 질문과 정답 SQL
question = "전체 직원 수는 몇 명인가요?"
expected_sql = "SELECT COUNT(*) as count FROM employees"

# SQLAgent로 SQL 생성
response = agent.query(question)
generated_sql = response.metadata.get("sql", "")

print("=== SQLAgent 테스트 ===")
print(f"질문: {question}")
print(f"\n정답 SQL: {expected_sql}")
print(f"생성된 SQL: {generated_sql}")

# 두 SQL 실행
expected_result, _ = db.execute_query(expected_sql)
generated_result, _ = db.execute_query(generated_sql)

print(f"\n정답 결과: {expected_result}")
print(f"생성 결과: {generated_result}")

# Execution Accuracy 판정
is_correct = compare_results(expected_result, generated_result)
print(f"\n=== Execution Accuracy: {is_correct} ===")

## 8. 정리 및 다음 단계

### 핵심 개념 정리

| 개념 | 설명 |
|------|------|
| **Execution Accuracy** | SQL 실행 결과가 같으면 정답으로 판정 |
| **Exact Match** | SQL 텍스트가 완전히 같아야 정답 (너무 엄격) |
| **왜 RAGAS가 아닌가?** | SQL은 실행 결과로 정확히 판단 가능 |

### 학습 완료 체크리스트

- [ ] RAG vs SQL 평가의 차이 이해
- [ ] Execution Accuracy 개념 이해
- [ ] 결과 비교 함수 원리 이해
- [ ] SQLAgent와 연동 테스트 완료

### 다음 단계

**step_02_sql_evaluation.ipynb**: 실제 구현

- Base 모델 vs Fine-tuned 모델 비교
- 전체 테스트셋 평가
- 카테고리별 분석
- 결과 저장

---

**수고하셨습니다!**

이제 Execution Accuracy가 무엇인지, 왜 SQL 평가에 적합한지 이해했습니다.